In [1]:
print("Jay Ganesh")

Jay Ganesh


In [2]:
import os
%pwd

'd:\\ProjectAarya\\MLOPs\\Code\\Ch21 NLP project with hugging face and transformer\\TextSummarization\\research'

In [ ]:
# os.chdir("..")
# %pwd

'd:\\ProjectAarya\\MLOPs\\Code\\Ch21 NLP project with hugging face and transformer\\TextSummarization'

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelTrainerConfig:
    root_dir: Path
    data_path:Path
    model_ckpt: Path
    num_train_epoch: int
    warmup_steps: int
    per_device_train_batch_size: int
    weight_decay: float
    logging_steps: int
    evaluation_strategy: str
    eval_steps: int
    save_steps: int
    gradient_accumulation_steps: int

In [5]:
from src.TextSummarizer.constants import *
from src.TextSummarizer.utils.common import read_yaml, create_directories

In [12]:
class ConfigurationManager:
    def __init__(
            self,
            config_file_path=CONFIG_FILE_PATH,
            params_file_path=PARAM_FILE_PATH,):
        
        self.config=read_yaml(config_file_path)
        self.params=read_yaml(params_file_path)
        create_directories([self.config.artifacts_root])
    
    def get_model_trainer_config(self)-> ModelTrainerConfig:
        config=self.config.model_trainer
        params=self.params.TrainingArguments
        model_trainer_config=ModelTrainerConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_ckpt=config.model_ckpt,
            num_train_epoch= params.num_train_epoch,
            warmup_steps= params.warmup_steps,
            per_device_train_batch_size= params.per_device_train_batch_size,
            weight_decay= params.weight_decay,
            logging_steps= params.logging_steps,
            evaluation_strategy= params.evaluation_strategy,
            eval_steps= params.eval_steps,
            save_steps=1e6,
            gradient_accumulation_steps= params.gradient_accumulation_steps  
        )
        return model_trainer_config

In [7]:
from transformers import AutoModelForSeq2SeqLM,AutoTokenizer
from transformers import DataCollatorForSeq2Seq
from transformers import TrainingArguments, Trainer

d:\ProjectAarya\MLOPs\Code\Ch21 NLP project with hugging face and transformer\TextSummarization\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
import torch

In [9]:
import torch
from datasets import load_from_disk

class ModelTrainer:
    def __init__(self,config:ModelTrainerConfig):
        self.config=config

    def train(self):  
        device="cuda" if torch.cuda.is_available() else "cpu"
        # device
        tokenizer=AutoTokenizer.from_pretrained(self.config.model_ckpt)
        model_pegasus=AutoModelForSeq2SeqLM.from_pretrained(self.config.model_ckpt).to(device)
        seq2seq_data_collator=DataCollatorForSeq2Seq(tokenizer,model=model_pegasus)

        #loading the data

        dataset_samsum_pt=load_from_disk(self.config.data_path)
        trainer_args=TrainingArguments(
            output_dir=self.config.root_dir,num_train_epochs=self.config.num_train_epoch,
            warmup_steps=self.config.warmup_steps, per_device_train_batch_size=self.config.per_device_train_batch_size,
            per_device_eval_batch_size=1,weight_decay=self.config.weight_decay,logging_steps=self.config.logging_steps,
            eval_strategy=self.config.evaluation_strategy,eval_steps=self.config.eval_steps,save_steps=self.config.save_steps,
            gradient_accumulation_steps=self.config.gradient_accumulation_steps

        )

        trainer=Trainer(model=model_pegasus,args=trainer_args,
                data_collator=seq2seq_data_collator,train_dataset=dataset_samsum_pt["test"],
                eval_dataset=dataset_samsum_pt["validation"])
        trainer.train()

        model_pegasus.save_pretrained("pegasus-samsum-model")
        tokenizer.save_pretrained("tokenizer")

In [10]:
# !pip install --upgrade accelerate
# !pip uninstall -y transformers accelerate
# !pip install transformers accelerate

In [ ]:
config=ConfigurationManager()
model_trainer_config=config.get_model_trainer_config()
model_trainer=ModelTrainer(config=model_trainer_config)
model_trainer.train()

ERROR! Session/line number was not unique in database. History logging moved to new session 424
[2026-04-08 17:00:19,470]:INFO common yaml file: config\config.yaml loaded and returned successfully
[2026-04-08 17:00:19,653]:INFO common yaml file: params.yaml loaded and returned successfully
[2026-04-08 17:00:19,704]:INFO common artifacts created successfully
[2026-04-08 17:00:20,185]:INFO _client HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
[2026-04-08 17:00:20,220]:INFO _client HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK"
[2026-04-08 17:00:20,653]:INFO _client HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
[2026-04-08 17:00:20,685]:INFO _client HTTP Request: HEAD https://huggingface.co/api/r

Loading weights: 100%|██████████| 680/680 [00:00<00:00, 1076.45it/s]
PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-cnn_dailymail
Key                                  | Status  | 
-------------------------------------+---------+-
model.encoder.embed_positions.weight | MISSING | 
model.decoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[2026-04-08 17:01:45,108]:INFO _client HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
[2026-04-08 17:01:45,160]:INFO _client HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/generation_config.json "HTTP/1.1 200 OK"


d:\ProjectAarya\MLOPs\Code\Ch21 NLP project with hugging face and transformer\TextSummarization\venv\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
